# l3d-bench: one baseline run (TopoNet, A2ONet, D2GPLand, BCRNet)

**Before running** (once per notebook): Accelerator **GPU T4 x2** · Internet **on** ·
Add data `khoatrytopublish/l3d-train`, `l3d-val`, `l3d-test` · Add-ons > Secrets: tick `RCLONE_CONFIG_GDRIVE`

Set the variables below, then Save Version > Save & Run All.
- The repo is cloned at its pinned commit and trained with the authors' own recipe; only our documented patches are applied (resume across sessions, an extra checkpoint chosen on val).
- `paused` at the end of the log = Kaggle's time limit: run the same notebook again, it continues from Drive.
- `done` = predictions exported and scored with our metric; results in `gdrive:l3d-bench/results/<run id>/`.

**Order:** first `PILOT = True` once per method (a few epochs on 8 frames; it pauses after epoch 1 on purpose and resumes, so setup, GPU memory and resume are all checked), then the real run.
**BCRNet:** the first minutes of every session build its CUDA op and compute SAM features + Bezier targets for all frames (about 10-15 min); that is expected.
**G2:** `METHOD = "toponet"`, `RELEASED = True` scores TopoNet's released weights (about 30 min, no training).

In [ ]:
METHOD = "toponet"        # toponet | a2onet | d2gpland | bcrnet
SPLIT = "official"        # official | f0 ... f4 (folds need data/splits/folds_5.json)
RELEASED = False          # True: TopoNet released weights only (gate G2)
PILOT = False             # True: 8 train frames, a few epochs, saved under _pilot__...
FORCE = False             # True: re-score a finished run from its checkpoints on Drive (no retraining)
BATCH_SIZE = None         # only if TopoNet's pilot runs out of GPU memory: 4 -> 2 (logged as a deviation).
                          # A2ONet uses both T4s (batch 1 per GPU, as in the paper); D2GPLand and BCRNet must stay at 4.
COMMIT = "main"

In [ ]:
import os, subprocess
REPO, WORK = "https://github.com/KhoaLeQuangMinh/l3d-bench.git", "/kaggle/working/l3d-bench"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "-q", REPO, WORK], check=True)
subprocess.run(["git", "-C", WORK, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", WORK, "checkout", "-q", COMMIT], check=True)
subprocess.run(["git", "-C", WORK, "pull", "-q", "--ff-only"], check=False)
os.chdir(WORK)
subprocess.run("pip install -q medpy && apt-get -qq update && apt-get -qq install -y rclone cmake > /dev/null", shell=True, check=True)
if BATCH_SIZE:
    os.environ["L3D_BATCH_SIZE"] = str(BATCH_SIZE)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
flags = ("--released " if RELEASED else "") + ("--pilot " if PILOT else "") + ("--force" if FORCE else "")
!python -m l3d_bench.baselines.run --method {METHOD} --split {SPLIT} {flags}